# Experiment 35 — Full backward-free SparseWalker

This is the first **from-scratch** version: no pretrained neural checkpoint, no optimizer, no `loss.backward()`, and no autograd graph during learning.

The model keeps fixed random item codes as stable identifiers. Router, concept values, graph query/keys, graph topology, and message readout all adapt using local forward-only rules driven by the observed next item.

**Beauty references under the same internal protocol:** SASRec FullCE test NDCG@10 = **0.03120**; backprop SparseWalker v1.1 = **0.04488**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/Sparsewalker
!git clone -q --depth 1 --branch agent/backward-free-walker https://github.com/hanialshater/Sparsewalker- /content/Sparsewalker
!pip install -q -e /content/Sparsewalker
%cd /content/Sparsewalker


In [ ]:
import torch, inspect, sys
sys.path.insert(0, '/content/Sparsewalker/experiments')
import run_amazon_backward_free_full as bf
print('torch', torch.__version__)
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
src = inspect.getsource(bf.backward_free_full_epoch)
assert '.backward(' not in src
assert 'torch.optim' not in src
print('STRICT_GATE', {'backward_calls_in_learning_loop': 0, 'optimizer_in_learning_loop': False})


## Run Beauty

The important trajectory is `BFFULL_INIT -> BFFULL_EPOCH -> BFFULL_RESULT`. The first run is intentionally about **whether local forward-only learning can produce useful ranking quality**, not training speed.

Interpretation guide for test NDCG@10: `<0.02` not working yet; `~0.03` reaches SASRec territory; `>=0.04` is very strong; `~0.045` approaches the normal backprop Walker.


In [ ]:
!python experiments/run_amazon_backward_free_full.py \
  --dataset beauty \
  --seed 42 \
  --epochs 30 \
  --batch-size 512 \
  --eval-every 1


In [ ]:
import json, pathlib
p = pathlib.Path('/content/drive/MyDrive/sparsewalker_backward_free_full/beauty/seed42/result.json')
result = json.loads(p.read_text())
result


In [ ]:
bf_ndcg = result['best_backward_free']['test']['NDCG@10']
sas = 0.031195719394901355
walker = 0.044882819399656555
print({
    'backward_free_test_NDCG@10': bf_ndcg,
    'vs_SASRec': bf_ndcg / sas,
    'vs_backprop_Walker': bf_ndcg / walker,
})
